In [1]:
import sqlite3
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import AgglomerativeClustering
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_distances

In [52]:
def get_table_names():
    conn = sqlite3.connect('seed.sqlite')
    cursor = conn.cursor()

    cursor.execute('SELECT name FROM sqlite_master WHERE type="table"')
    results = cursor.fetchall()

    print('table results:', results)

    for table in results:
        table_name = table[0]
        cursor.execute(f'SELECT * FROM {table_name}')
        cols = [desc[0] for desc in cursor.description]
        print(f'\nTable: {table_name}')
        print('Columns:', cols)

    conn.close()
    return

def get_titles(date='2026-07-20'):
    conn = sqlite3.connect('seed.sqlite')
    cursor = conn.cursor()

    # Use '?' for the date, and filter out missing embeddings
    query = '''
        SELECT title, feed_source, summary, embedding 
        FROM items 
        WHERE date(published_at) = ? AND embedding IS NOT NULL
    '''
    cursor.execute(query, (date,))
    results = cursor.fetchall()

    conn.close()
    return results

## Embed new items

In [53]:
from sentence_transformers import SentenceTransformer
from bs4 import BeautifulSoup

MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 64


def clean_summary(summary):
    if not summary:
        return ""
    return BeautifulSoup(summary, "html.parser").get_text(" ", strip=True)


def build_text(title, summary):
    title = title or ""
    summary = clean_summary(summary)
    return f"{title}. {title}. {summary}".strip()


def embed_new_items():
    model = SentenceTransformer(MODEL_NAME)
    conn = sqlite3.connect("seed.sqlite")
    cursor = conn.cursor()

    cursor.execute(
        "SELECT id, title, summary FROM items WHERE embedding IS NULL"
    )
    rows = cursor.fetchall()
    print(f"{len(rows)} items to embed")

    if not rows:
        conn.close()
        return

    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        texts = [build_text(t, s) for _, t, s in batch]

        vecs = model.encode(
            texts,
            normalize_embeddings=True,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
        ).astype(np.float32)

        cursor.executemany(
            "UPDATE items SET embedding = ?, embedding_model = ? WHERE id = ?",
            [(v.tobytes(), MODEL_NAME, row[0]) for v, row in zip(vecs, batch)],
        )
        conn.commit()
        print(f"  {start + len(batch)}/{len(rows)}")

    conn.close()


if __name__ == "__main__":
    embed_new_items()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2964.52it/s]


KeyboardInterrupt: 

## Taggign system

In [ ]:
get_table_names()

table results: [('items',), ('sqlite_sequence',), ('item_tags',)]

Table: items
Columns: ['id', 'feed_source', 'url', 'title', 'summary', 'categories', 'published_at', 'fetched_at', 'read', 'note', 'discord_message_id', 'score', 'rank', 'digest_date', 'embedding', 'embedding_model']

Table: sqlite_sequence
Columns: ['name', 'seq']

Table: item_tags
Columns: ['item_id', 'tag']


In [ ]:
import sqlite3

conn = sqlite3.connect("seed.sqlite")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS item_tags")

cursor.execute("""
    CREATE TABLE tags (
        id             INTEGER PRIMARY KEY AUTOINCREMENT,
        name           TEXT NOT NULL UNIQUE,
        centroid       BLOB NOT NULL,
        weight         REAL NOT NULL DEFAULT 1.0,
        article_count  INTEGER NOT NULL DEFAULT 0,
        created_at     TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
        last_seen_at   TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP
    )
""")

cursor.execute("""
    CREATE TABLE item_tags (
        item_id     INTEGER NOT NULL,
        tag_id      INTEGER NOT NULL,
        similarity  REAL NOT NULL,
        PRIMARY KEY (item_id, tag_id),
        FOREIGN KEY (item_id) REFERENCES items(id) ON DELETE CASCADE,
        FOREIGN KEY (tag_id)  REFERENCES tags(id)  ON DELETE CASCADE
    )
""")

cursor.execute("CREATE INDEX idx_item_tags_item_id ON item_tags(item_id)")
cursor.execute("CREATE INDEX idx_item_tags_tag_id ON item_tags(tag_id)")

conn.commit()
conn.close()
print("done")

done


In [ ]:
import sqlite3

conn = sqlite3.connect("seed.sqlite")
cursor = conn.cursor()

for table in ("tags", "item_tags"):
    cursor.execute(f"PRAGMA table_info({table})")
    print(f"\n{table}:")
    for col in cursor.fetchall():
        print(f"  {col[1]:15} {col[2]:10} {'NOT NULL' if col[3] else ''}")

cursor.execute("PRAGMA index_list(item_tags)")
print("\nitem_tags indexes:", cursor.fetchall())

conn.close()


tags:
  id              INTEGER    
  name            TEXT       NOT NULL
  centroid        BLOB       NOT NULL
  weight          REAL       NOT NULL
  article_count   INTEGER    NOT NULL
  created_at      TIMESTAMP  NOT NULL
  last_seen_at    TIMESTAMP  NOT NULL

item_tags:
  item_id         INTEGER    NOT NULL
  tag_id          INTEGER    NOT NULL
  similarity      REAL       NOT NULL

item_tags indexes: [(0, 'idx_item_tags_tag_id', 0, 'c', 0), (1, 'idx_item_tags_item_id', 0, 'c', 0), (2, 'sqlite_autoindex_item_tags_1', 1, 'pk', 0)]


In [ ]:
import sqlite3

conn = sqlite3.connect("seed.sqlite")
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM items")
total = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM items WHERE embedding IS NOT NULL")
embedded = cursor.fetchone()[0]

conn.close()

print(f"total items:    {total}")
print(f"with embedding: {embedded}")
print(f"missing:        {total - embedded}")

total items:    2964
with embedding: 2964
missing:        0


### Resetting tags

In [ ]:
# import sqlite3

# conn = sqlite3.connect("seed.sqlite")
# cursor = conn.cursor()

# cursor.execute("DELETE FROM item_tags")
# cursor.execute("DELETE FROM tags")
# cursor.execute("DELETE FROM sqlite_sequence WHERE name IN ('tags', 'item_tags')")

# conn.commit()

# for table in ("tags", "item_tags"):
#     cursor.execute(f"SELECT COUNT(*) FROM {table}")
#     print(f"{table}: {cursor.fetchone()[0]} rows")

# conn.close()

tags: 0 rows
item_tags: 0 rows


### Resetting embeds

In [ ]:
# import sqlite3

# conn = sqlite3.connect("seed.sqlite")
# cursor = conn.cursor()

# cursor.execute("SELECT COUNT(*) FROM items WHERE embedding IS NOT NULL")
# before = cursor.fetchone()[0]

# cursor.execute("UPDATE items SET embedding = NULL, embedding_model = NULL")
# conn.commit()

# cursor.execute("SELECT COUNT(*) FROM items WHERE embedding IS NOT NULL")
# after = cursor.fetchone()[0]

# conn.close()

# print(f"cleared {before - after} embeddings ({after} remaining)")

cleared 2964 embeddings (0 remaining)


In [57]:
import sqlite3
import ollama
from bs4 import BeautifulSoup

MODEL = "qwen2.5:1.5b"
VALID_TAGS = [
    "politics", "world affairs", "business", "technology", "science",
    "health", "environment", "culture", "sports", "society",
    "history", "lifestyle", "other",
]


def clean(s):
    return BeautifulSoup(s or "", "html.parser").get_text(" ", strip=True)


def tag_article(title, summary):
    text = f"{title or ''}. {clean(summary)}"[:1500]

    pick_prompt = (
        f"You are a news tagger. Choose up to 3 tags from this list that best fit the article. "
        f"Prefer fewer tags if only 1 or 2 clearly apply. "
        f"Return only tag names as a comma-separated list, nothing else.\n\n"
        f"Valid tags: {', '.join(VALID_TAGS)}\n\n"
        f"Article: {text}\n\n"
        f"Tags:"
    )
    picked_raw = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": pick_prompt}],
        options={"temperature": 0.0},
    )["message"]["content"].strip()

    picked = [t.strip().lower() for t in picked_raw.split(",")]
    picked = [t for t in picked if t in VALID_TAGS][:3]  # hard cap

    if not picked:
        return picked_raw, []

    scored = []
    for tag in picked:
        score_prompt = (
            f"On a scale of 1 to 3, how well does this article fit the tag '{tag}'?\n"
            f"1 = weak fit, 2 = moderate fit, 3 = strong fit.\n"
            f"Return only the number.\n\n"
            f"Article: {text}"
        )
        score_raw = ollama.chat(
            model=MODEL,
            messages=[{"role": "user", "content": score_prompt}],
            options={"temperature": 0.0},
        )["message"]["content"].strip()

        try:
            n = int("".join(c for c in score_raw if c.isdigit())[:1])
            if n in (1, 2, 3):
                scored.append((tag, n))
            else:
                scored.append((tag, None))
        except ValueError:
            scored.append((tag, None))

    return picked_raw, scored


# Sample 100 items
conn = sqlite3.connect("seed.sqlite")
rows = conn.execute("""
    SELECT id, title, summary FROM items
    WHERE embedding IS NOT NULL
    ORDER BY RANDOM() LIMIT 100
""").fetchall()
conn.close()

print(f"tagging {len(rows)} items with {MODEL}\n")
for i, (item_id, title, summary) in enumerate(rows, 1):
    raw, scored = tag_article(title, summary)
    tag_str = ", ".join(f"{t}({s})" for t, s in scored) if scored else f"NO VALID TAGS (raw: {raw!r})"
    print(f"[{i:>3}] {(title or '')[:75]}")
    print(f"      → {tag_str}\n")

tagging 100 items with qwen2.5:1.5b

[  1] Forget EVs. Cycling is revolutionising transport
      → technology(3)

[  2] Brutal Humanoid Robot Cage Match Stops Abruptly When One of the Robot’s Hea
      → technology(3), science(3), world affairs(3)

[  3] reMarkable’s new Paper Pure is good. That’s why I wrote this review on it.
      → technology(3), business(2), science(3)

[  4] The Zoom hack that says, ‘Don’t record me’
      → technology(2), business(2), world affairs(2)

[  5] The new front in China’s cyber campaign against America
      → technology(3), politics(3), world affairs(3)

[  6] Why Energy Is the New Global Arms Race
      → technology(3), politics(2), world affairs(3)

[  7] How Outdated Budget Rules Are Holding Back American Industrial Policy
      → technology(3), business(2), politics(2)

[  8] US attacks Iran for 10th consecutive night
      → technology(2), politics(3), world affairs(3)

[  9] Trump Shrinks Utah National Monuments by 90%
      → politics(3), wor

In [2]:
HARD_CASES = [
    (2,  "Brutal Humanoid Robot Cage Match Stops Abruptly When One of the Robot's Head Falls Off",
         "Two humanoid robots fought in a boxing match that ended when one lost its head."),
    (8,  "US attacks Iran for 10th consecutive night",
         "American forces continued strikes on Iranian targets overnight."),
    (10, "More Canadian wildfire smoke shrouds US midwest, mid-Atlantic and north-east",
         "Air quality alerts issued across multiple US states as Canadian wildfire smoke drifts south."),
    (17, "The Mini Crossword: Thursday, July 16, 2026",
         "Today's Mini crossword puzzle."),
    (20, "Korea's robot umpires reduce favorable calls for star players",
         "Automated ball-strike systems in Korean baseball are producing more consistent calls regardless of the batter's stature."),
    (26, "Ossoff: Trump 'signaling his plans to attack the election'",
         "Senator Ossoff warned that recent rhetoric from Trump suggests he intends to challenge election results."),
    (30, "Poop From Southern Right Whales Reveals That the Animals Might Be More Resilient",
         "Analysis of whale scat suggests the population is adapting better than expected to changing ocean conditions."),
    (40, "Third person dies amid NYC Legionnaires' outbreak, officials say",
         "New York City health officials confirmed a third death from the outbreak in a Bronx neighbourhood."),
    (45, "5.5 magnitude earthquake hits Peru's Andes region and kills at least 6 people",
         "A moderate earthquake struck the Peruvian Andes, killing several and damaging homes."),
    (52, "Ancient DNA analysis reveals Wiltshire's Upton Lovell Shaman was a woman",
         "Genetic analysis of Bronze Age remains upends assumptions about the individual's identity."),
    (61, "Argentina Incensed After UK Deploys Ship to South America",
         "Argentina has protested the Royal Navy's deployment of a warship near the Falkland Islands."),
    (75, "Show HN: IKEA Complexity Index",
         "A hobby project that ranks IKEA furniture by how hard it is to assemble."),
    (82, "White House backs Argentina players over Falklands banner in World Cup semi-final",
         "The White House issued a statement supporting Argentine players who displayed a Falklands banner during the World Cup semi-final."),
    (86, "Why did Henry VIII boil his cook alive? 17 grisly tales from Tudor England",
         "Historians explore the strangest and most brutal executions of the Tudor period."),
    (97, "Luke Skywalker's Lightsaber and Severed Hand From Famed Darth Vader Duel Sold at Auction",
         "The screen-used props from The Empire Strikes Back sold at auction for millions."),
]

In [6]:
import ollama
import time
import re


MODEL = "gemma3:4b"
VALID_TAGS = [
    "politics", "world affairs", "business", "technology", "science",
    "health", "environment", "culture", "sports", "society",
    "history", "lifestyle", "other",
]

WORD_TO_SCORE = {"CENTRAL": 3, "MAJOR": 2, "MINOR": 1}



def tag_article_combined(title, summary):
    text = f"{title or ''}. {summary or ''}"[:1500]

    prompt = f"""Choose up to 3 tags from this list that describe this article, and rate each one.

Valid tags: {', '.join(VALID_TAGS)}

Ratings:
- CENTRAL: article is fundamentally about this topic
- MAJOR: one of several genuine themes
- MINOR: only touched on briefly

Rules:
- Most articles need only 1 tag.
- Never add tags that only tangentially apply.
- You MUST include a rating for every tag you pick.
- If nothing on the list fits, return only: tag: other / rating: MINOR

Article: {text}

Respond in this exact format, alternating tag and rating on separate lines:
tag: <tag name>
rating: <CENTRAL, MAJOR, or MINOR>
tag: <tag name>
rating: <CENTRAL, MAJOR, or MINOR>"""

    raw = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.0, "num_predict": 120},
        keep_alive="30m",
    )["message"]["content"].strip()

    # Extract all tag/rating pairs by scanning lines
    scored = []
    current_tag = None
    for line in raw.splitlines():
        line = line.strip().strip("*").strip("-").strip()
        m_tag = re.match(r"^tag\s*:\s*(.+)$", line, re.IGNORECASE)
        m_rating = re.match(r"^rating\s*:\s*(.+)$", line, re.IGNORECASE)

        if m_tag:
            candidate = m_tag.group(1).strip().lower().strip('"').strip("'")
            if candidate in VALID_TAGS:
                current_tag = candidate
            else:
                current_tag = None
        elif m_rating and current_tag:
            rating_text = m_rating.group(1).strip().upper()
            score = None
            for word, val in WORD_TO_SCORE.items():
                if word in rating_text:
                    score = val
                    break
            if score is not None:
                scored.append((current_tag, score))
            current_tag = None
        if len(scored) >= 3:
            break

    return raw, scored


print(f"tagging {len(HARD_CASES)} hard cases with {MODEL} — combined call\n")
total = 0
for orig_id, title, summary in HARD_CASES:
    t0 = time.time()
    raw, scored = tag_article_combined(title, summary)
    dt = time.time() - t0
    total += dt
    tag_str = ", ".join(f"{t}({s})" for t, s in scored) if scored else f"NO VALID TAGS (raw: {raw!r})"
    print(f"[{orig_id:>3}] ({dt:>4.1f}s) {title[:70]}")
    print(f"      → {tag_str}\n")

print(f"total: {total:.1f}s  ({total/len(HARD_CASES):.1f}s/article)")

tagging 15 hard cases with gemma3:4b — combined call

[  2] (32.1s) Brutal Humanoid Robot Cage Match Stops Abruptly When One of the Robot'
      → technology(3), sports(2), other(1)

[  8] (28.3s) US attacks Iran for 10th consecutive night
      → world affairs(3), politics(2)

[ 10] (29.6s) More Canadian wildfire smoke shrouds US midwest, mid-Atlantic and nort
      → environment(3), world affairs(2), health(1)

[ 17] (28.4s) The Mini Crossword: Thursday, July 16, 2026
      → culture(1), lifestyle(1), other(3)

[ 20] (29.7s) Korea's robot umpires reduce favorable calls for star players
      → sports(3), technology(2), business(1)

[ 26] (31.5s) Ossoff: Trump 'signaling his plans to attack the election'
      → politics(3), world affairs(2), society(1)

[ 30] (33.1s) Poop From Southern Right Whales Reveals That the Animals Might Be More
      → science(3), environment(2), health(1)

[ 40] (32.4s) Third person dies amid NYC Legionnaires' outbreak, officials say
      → health(3), envi

In [1]:
import ollama
import time
import re

MODEL = "gemma3:4b" # Ensure this matches your pulled model
VALID_TAGS = [
    "politics", "world affairs", "business", "technology", "science",
    "health", "environment", "culture", "sports", "society",
    "history", "lifestyle", "other",
]
WORD_TO_SCORE = {"CENTRAL": 3, "MAJOR": 2, "MINOR": 1}

# Dummy data so you can test this immediately
HARD_CASES = [
    ("001", "Global Markets Rally", "Stocks surged today following unexpected policy shifts..."),
    ("002", "New CRISPR Breakthrough", "Scientists have successfully edited a gene responsible for..."),
    ("003", "Local Mayor Resigns", "City hall was thrown into chaos today as the mayor..."),
    ("004", "Tech Giants Merge", "In a surprise move, two of the largest tech companies..."),
    ("005", "World Cup Final", "The championship match ended in a thrilling penalty shootout...")
]

def tag_article_combined(title, summary):
    text = f"{title or ''}. {summary or ''}"[:1500]

    prompt = f"""Choose up to 3 tags from this list that describe this article, and rate each one.

Valid tags: {', '.join(VALID_TAGS)}

Ratings:
- CENTRAL: article is fundamentally about this topic
- MAJOR: one of several genuine themes
- MINOR: only touched on briefly

Rules:
- Most articles need only 1 tag.
- Never add tags that only tangentially apply.
- You MUST include a rating for every tag you pick.
- If nothing on the list fits, return only: tag: other / rating: MINOR

Article: {text}

Respond in this exact format, alternating tag and rating on separate lines:
tag: <tag name>
rating: <CENTRAL, MAJOR, or MINOR>
"""

    # Capture the entire response object to get the telemetry
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.0, "num_predict": 120},
        keep_alive="30m",
    )
    
    raw = response["message"]["content"].strip()
    
    # Ollama returns exact timings in nanoseconds — convert to seconds
    load_time = response.get("load_duration", 0) / 1e9
    eval_time = response.get("eval_duration", 0) / 1e9
    eval_count = response.get("eval_count", 0)

    # Extract all tag/rating pairs
    scored = []
    current_tag = None
    for line in raw.splitlines():
        line = line.strip().strip("*").strip("-").strip()
        m_tag = re.match(r"^tag\s*:\s*(.+)$", line, re.IGNORECASE)
        m_rating = re.match(r"^rating\s*:\s*(.+)$", line, re.IGNORECASE)

        if m_tag:
            candidate = m_tag.group(1).strip().lower().strip('"').strip("'")
            if candidate in VALID_TAGS:
                current_tag = candidate
            else:
                current_tag = None
        elif m_rating and current_tag:
            rating_text = m_rating.group(1).strip().upper()
            score = None
            for word, val in WORD_TO_SCORE.items():
                if word in rating_text:
                    score = val
                    break
            if score is not None:
                scored.append((current_tag, score))
            current_tag = None
        if len(scored) >= 3:
            break

    return raw, scored, load_time, eval_time, eval_count


print(f"=== WARMING UP {MODEL} ===")
# A tiny dummy call forces the model into memory and pays the load penalty once
ollama.chat(model=MODEL, messages=[{"role": "user", "content": "hi"}], keep_alive="30m")
print("Warm-up complete. Model is now loaded.\n")


print(f"=== TAGGING {len(HARD_CASES)} CASES ===\n")
total_wall = 0

for orig_id, title, summary in HARD_CASES:
    t0 = time.time()
    raw, scored, load_time, eval_time, eval_count = tag_article_combined(title, summary)
    wall_time = time.time() - t0
    
    total_wall += wall_time
    
    # Calculate generation speed (Tokens Per Second)
    tps = (eval_count / eval_time) if eval_time > 0 else 0
    
    tag_str = ", ".join(f"{t}({s})" for t, s in scored) if scored else "NO VALID TAGS"
    
    print(f"[{orig_id:>3}] Wall: {wall_time:>4.1f}s | Load: {load_time:>4.2f}s | Eval: {eval_time:>4.2f}s ({tps:>4.1f} t/s)")
    print(f"      → {tag_str}\n")

print(f"Total Wall Time: {total_wall:.1f}s  (Avg: {total_wall/len(HARD_CASES):.1f}s/article)")

=== WARMING UP gemma3:4b ===
Warm-up complete. Model is now loaded.

=== TAGGING 5 CASES ===

[001] Wall:  7.7s | Load: 1.08s | Eval: 3.07s ( 8.5 t/s)
      → business(3), world affairs(2), politics(1)

[002] Wall:  5.4s | Load: 1.02s | Eval: 2.95s ( 8.5 t/s)
      → science(3), technology(2), health(1)

[003] Wall:  3.3s | Load: 0.91s | Eval: 0.93s ( 9.6 t/s)
      → politics(3)

[004] Wall:  5.3s | Load: 0.91s | Eval: 2.90s ( 9.0 t/s)
      → business(2), technology(3), politics(1)

[005] Wall:  3.7s | Load: 1.10s | Eval: 0.97s ( 9.3 t/s)
      → sports(3)

Total Wall Time: 25.3s  (Avg: 5.1s/article)
